In [0]:
# ===========================================
# Notebook Name:
# 03_migrate_existing_data
#
# Purpose:
# Migrate existing table contents into v2 shape.
# Most v2 tables are passthrough (no data movement,
# structure already matches). Two tables need a
# real, idempotent data migration:
#
# 1. silver.tournaments column consolidation
#    (COALESCE new columns from legacy columns)
# 2. bronze.tournament_list_scrape_log ->
#    bronze.scrape_log (coarse 1-run=1-row copy,
#    source_type='tournament_list_run')
#
# This notebook does not drop any legacy columns
# or tables. Deletion is gated on Step10 criteria
# (see docs/migration_plan section 14).
# ===========================================

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.dropdown(
    "force_remigrate", "false", ["false", "true"]
)

FORCE_REMIGRATE = (
    dbutils.widgets.get("force_remigrate") == "true"
)

print("force_remigrate:", FORCE_REMIGRATE)

In [0]:
# -------------------------------------------
# Passthrough tables: structure already matches
# v2 target shape. No data movement performed.
# Listed here only for traceability.
# -------------------------------------------
PASSTHROUGH_TABLES = [
    "pokemon.bronze.tournament_list_raw",
    "pokemon.bronze.event_result_raw",
    "pokemon.bronze.deck_raw",
    "pokemon.silver.deck_cards",
    "pokemon.silver.decks",
    "pokemon.silver.tournament_results",
    "pokemon.gold.card_usage",
    "pokemon.gold.deck_registry",
    "pokemon.gold.deck_pokemon_features",
    "pokemon.gold.deck_similarity",
    "pokemon.gold.deck_archetypes",
    "pokemon.ops.result_fetch_targets",
]

for table_name in PASSTHROUGH_TABLES:
    exists = spark.catalog.tableExists(table_name)
    print(f"{table_name}: exists={exists} (passthrough, no migration)")

In [0]:
# -------------------------------------------
# Migration 1: silver.tournaments column
# consolidation.
#
# New (v2) columns win when present; legacy
# columns fill gaps via COALESCE. Legacy columns
# are left in place (not dropped) until Step10.
# -------------------------------------------
TOURNAMENTS_TABLE = "pokemon.silver.tournaments"

before_null_counts = spark.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN venue_name IS NULL THEN 1 ELSE 0 END) AS venue_name_nulls,
        SUM(CASE WHEN prefecture IS NULL THEN 1 ELSE 0 END) AS prefecture_nulls,
        SUM(CASE WHEN event_type IS NULL THEN 1 ELSE 0 END) AS event_type_nulls,
        SUM(CASE WHEN event_date IS NULL THEN 1 ELSE 0 END) AS event_date_nulls
    FROM {TOURNAMENTS_TABLE}
""")

display(before_null_counts)

In [0]:
spark.sql(f"""
    UPDATE {TOURNAMENTS_TABLE}
    SET
        venue_name = COALESCE(venue_name, venue),
        prefecture = COALESCE(prefecture, prefecture_name),
        event_type = COALESCE(event_type, event_type_title)
    WHERE
        venue_name IS NULL
        OR prefecture IS NULL
        OR event_type IS NULL
""")

print("silver.tournaments column consolidation applied.")

In [0]:
after_null_counts = spark.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN venue_name IS NULL THEN 1 ELSE 0 END) AS venue_name_nulls,
        SUM(CASE WHEN prefecture IS NULL THEN 1 ELSE 0 END) AS prefecture_nulls,
        SUM(CASE WHEN event_type IS NULL THEN 1 ELSE 0 END) AS event_type_nulls,
        SUM(CASE WHEN event_date IS NULL THEN 1 ELSE 0 END) AS event_date_nulls
    FROM {TOURNAMENTS_TABLE}
""")

display(after_null_counts)

In [0]:
# -------------------------------------------
# Migration 2: tournament_list_scrape_log ->
# scrape_log (coarse copy, one row per run).
#
# Idempotent: if force_remigrate is false and
# rows for source_type='tournament_list_run'
# already exist, skip. If true, delete then
# re-copy.
# -------------------------------------------
SCRAPE_LOG_TABLE = "pokemon.bronze.scrape_log"
LEGACY_LIST_LOG_TABLE = "pokemon.bronze.tournament_list_scrape_log"

already_migrated_count = spark.sql(f"""
    SELECT COUNT(*) AS cnt
    FROM {SCRAPE_LOG_TABLE}
    WHERE source_type = 'tournament_list_run'
""").collect()[0]["cnt"]

print("Already-migrated tournament_list_run rows:", already_migrated_count)

In [0]:
should_migrate = (
    already_migrated_count == 0
) or FORCE_REMIGRATE

print("should_migrate:", should_migrate)

if should_migrate and FORCE_REMIGRATE and already_migrated_count > 0:
    spark.sql(f"""
        DELETE FROM {SCRAPE_LOG_TABLE}
        WHERE source_type = 'tournament_list_run'
    """)
    print("Deleted existing tournament_list_run rows for re-migration.")

In [0]:
if should_migrate:
    legacy_list_log_df = spark.table(LEGACY_LIST_LOG_TABLE)

    migrated_df = (
        legacy_list_log_df
        .select(
            F.col("run_id"),
            F.lit("tournament_list_run").alias("source_type"),
            F.col("run_id").alias("source_id"),
            # request_url is NOT NULL on scrape_log; the legacy
            # per-run log has no single URL (paged), so record a
            # placeholder marking this as a migrated coarse row.
            F.lit("migrated:tournament_list_scrape_log").alias(
                "request_url"
            ),
            F.lit(None).cast("int").alias("http_status"),
            F.col("status"),
            F.col("elapsed_ms"),
            F.col("total_api_event_count").alias("records_found"),
            F.lit(None).cast("string").alias("error_type"),
            F.col("error_message"),
            F.col("finished_at").alias("scraped_at"),
            F.to_date(F.col("started_at")).alias("ingestion_date"),
        )
    )

    (
        migrated_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(SCRAPE_LOG_TABLE)
    )

    print(
        "Migrated",
        migrated_df.count(),
        "tournament_list_scrape_log rows into scrape_log.",
    )
else:
    print("Skipped: tournament_list_run rows already present in scrape_log.")

In [0]:
display(
    spark.table(SCRAPE_LOG_TABLE)
    .filter(F.col("source_type") == "tournament_list_run")
    .orderBy(F.col("scraped_at").desc())
)